# 04 — Drifter–satellite complementary sampling

**Question:** How does prioritising drifter observations in the inner fit affect the full ESP estimate when satellite observations supply outer coverage?

One selected day, four comparisons and two small weight sweeps. Default: **29 October 2023, 06:00 UTC**, the day illustrated in the original `LATTE_app2_multiplatform_data.ipynb` (index 10). First show the observations' radial coverage rather than assume that drifters constrain only the inner region.

Import original inner functions from `ESP_zonodo`; keep weighted functions locally. No external backend edits, thinning, translation adjustment, uncertainty estimation or time-window sweep. Restart the kernel and run in order.

In [ ]:
from pathlib import Path

SAVE_OUTPUTS = False  # display only; does not disable Jupyter saving cell outputs
ESP_ROOT = Path('/home/z5297792/ESP_zonodo')
DRIFTER_FILE = ESP_ROOT / 'data/drifter_data.nc'
SATELLITE_FILE = ESP_ROOT / 'data/satellite_data.nc'

ANALYSIS_TIME = '2023-10-29 06:00:00'  # original illustrated day; other daily 06 UTC dates allowed
HALF_WINDOW_HOURS = 24  # inclusive +/-1 day, as in original
DATA_CUTOFF = '2024-01-27 06:00:00'  # original archive truncation
LON0, LAT0 = 154.3, -37.9
CORE_RADIUS_M = 35_000.0  # original code default; reconcile with manuscript's 30 km description
OUTER_RADIUS_M = 100_000.0
RC_LIMIT_M = 1_000_000.0  # numerical safeguard; not the observation-selection radius

SAT_WEIGHT_INNER = 0.0
SAT_WEIGHT_OUTER = 1.0
WEIGHT_SWEEP = [0.0, 0.1, 0.3, 1.0]
OUTPUT_ROOT = Path('results/04_drifter_satellite_weighting')

For each stage, use the same platform-normalised objective as notebook 3:

\[
J=\operatorname{mean}(e_D^2)+\lambda\operatorname{mean}(e_S^2).
\]

Inner errors are two-component **linear-core** velocity residuals; outer errors are **normalised tangential Gaussian-profile** residuals. The drifter platform multiplier is 1. `lambda=0` excludes satellite; `lambda=1` gives equal platform-average terms. Equal per-observation weighting corresponds to `N_satellite/N_drifter`, separately for each stage after applicable filters.

These are explicit influence choices, not inverse observation-error variances. They remove between-platform total-count scaling, not within-platform clustering/correlation or differences between nominal 15 m drifter motion and surface geostrophic flow. In situ does not automatically mean more accurate for the geostrophic eddy component. All drifter IDs in the window are pooled as one platform; report their number below.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import validation_helpers as vh
import platform_weighting as pw
import drifter_weighting as dw

backend, backend_info = vh.load_backend(ESP_ROOT)
drifters, satellite, data_info = dw.load_inputs(
    DRIFTER_FILE, SATELLITE_FILE, DATA_CUTOFF, LON0, LAT0)
track_settings = dict(core_radius=CORE_RADIUS_M, outer_radius=OUTER_RADIUS_M,
                      half_window_hours=HALF_WINDOW_HOURS)
observations, inner, core_ids, outer_ids, history = dw.track_to_day(
    backend, drifters, satellite, ANALYSIS_TIME, **track_settings)
reference = dw.finish_fit(observations, inner, core_ids, outer_ids,
                         'Original combined', original=True, rc_limit=RC_LIMIT_M)
if not reference['summary']['valid']:
    raise RuntimeError(f"Original combined outer fit failed: {reference['summary']}")

# Native satellite timestamps supporting this daily snapshot.
t = pd.Timestamp(ANALYSIS_TIME)
j = satellite['time'].searchsorted(t)
native_used = [satellite['time'][j]] if satellite['time'][j] == t else list(satellite['time'][j-1:j+1])
data_info.update(analysis_time=str(t), native_satellite_used=[str(x) for x in native_used])
print('Backend:', backend_info)
print('Native satellite timestamps (UTC):', data_info['native_satellite_used'])
print('Drifter window (UTC):', t-pd.Timedelta(hours=HALF_WINDOW_HOURS), 'to', t+pd.Timedelta(hours=HALF_WINDOW_HOURS))
display(observations.groupby('platform').agg(n=('id','size'),start=('time','min'),end=('time','max')))

**Baseline selection:** replay the original daily combined trajectory from the first satellite day through the selected day. Both inner passes use the previous day's ellipse; the centre changes between passes. This preserves the original implementation, including its lagged ellipse convention. Satellite-only follows that same rule independently. The original standalone drifter series ran at 12 UTC without a spatial core cut; our drifter-only comparison instead uses the **same 06 UTC window and fixed combined-core drifter subset** as the weighted fits. It is therefore a controlled inner comparison, not a reproduction of that separately timed series.

The weighted fits freeze the original combined inner and outer observation sets. Their effective radii may exceed a selection threshold once geometry changes; no new observations are added. The original outer rotation-sign filter remains active and retained counts may change. Satellite velocity is interpolated from native fields to the original daily 06 UTC grid, without temporal translation correction.

In [ ]:
evaluation_ids = sorted(set(core_ids) | set(outer_ids))
evaluation = observations[observations.id.isin(evaluation_ids)].copy()
core_evaluation = observations[observations.id.isin(core_ids)].copy()
coverage_table = dw.coverage(evaluation, reference)
display(coverage_table)
fig_coverage = dw.plot_coverage(evaluation, reference, CORE_RADIUS_M)
plt.show()

The radial plot uses **one reference ellipse and speed-peak radius**, from the original combined fit. It shows sampling relative to a fitted reference, not a known true boundary. Having observations on both sides of the peak is a support diagnostic, not a confidence interval or a sufficient condition for identifying the outer parameters.

In [ ]:
# Original inner estimator on the fixed core's drifter observations.
d = core_evaluation[core_evaluation.platform == 'Drifter']
drifter_inner = pw.backend_inner(backend, d)
drifter_fit = dw.finish_fit(observations, drifter_inner, d.id.tolist(), [],
    'Drifter only (inner)', original=True, inner_only=True)

_, sat_inner, sat_core, sat_outer, satellite_history = dw.track_to_day(
    backend, drifters, satellite, ANALYSIS_TIME, source='Satellite', **track_settings)
satellite_fit = dw.finish_fit(observations, sat_inner, sat_core, sat_outer,
    'Satellite only', original=True, rc_limit=RC_LIMIT_M)
selected = dw.weighted_fit(observations, reference, 'Selected weighted',
                          SAT_WEIGHT_INNER, SAT_WEIGHT_OUTER, RC_LIMIT_M)
comparisons = [drifter_fit, satellite_fit, reference, selected]

# Formulation checks against the actual external functions, not hard-coded field results.
checks = []
for f in [drifter_fit, satellite_fit, reference]:
    core = observations[observations.id.isin(f['inner_ids'])]
    counts = core.platform.value_counts()
    weight = counts.get('Satellite', 0) / counts.get('Drifter', 1)
    local = pw.weighted_inner(core, weight, reference_platform='Drifter')
    original = f['inner']
    np.testing.assert_allclose([local['xc'],local['yc']], [original['xc'],original['yc']],atol=.1,rtol=0)
    np.testing.assert_allclose(local['Q'],original['Q'],atol=1e-5,rtol=1e-5)
    np.testing.assert_allclose(local['Omega'],original['Omega'],atol=1e-10,rtol=1e-5)
    if f['summary']['valid']:
        outer = observations[observations.id.isin(f['outer_ids'])]
        rc, _, omega = backend.out_core_param_fit(
            *[outer[k].to_numpy() for k in ['x','y','u','v']],
            original['xc'],original['yc'],original['Q'],Omega0=original['Omega'],Rc_max=RC_LIMIT_M)
        np.testing.assert_allclose([f['summary']['Rc_km']*1000,f['summary']['Omega']],
                                  [rc,omega],rtol=1e-5,atol=1e-10)
    checks.append(dict(label=f['summary']['label'],inner_check='passed',
                       outer_check='passed' if f['summary']['valid'] else 'not tested'))
display(pd.DataFrame(checks))
columns = ['label','status','xc_km','yc_km','alpha','angle_deg','inner_vorticity',
           'Rc_km','vorticity','peak_bracketed']
comparison_table = pd.DataFrame([f['summary'] for f in comparisons])
display(comparison_table.reindex(columns=columns))
display(comparison_table.filter(regex=r'^label$|^n_'))
print('Linear-core residuals on the common core observations:')
display(dw.inner_residuals(comparisons, core_evaluation))
print('Full Gaussian residuals on the common evaluation observations:')
display(pw.residuals(comparisons, evaluation))

The drifter-only reference intentionally has no full-profile estimate. Inner vorticity uses the linear-core amplitude; final vorticity uses the amplitude refined by the Gaussian outer fit. Even with satellite inner weight zero, satellite outer observations can change the final vorticity.

Residuals overlap fitting data and describe agreement with each platform, **not independent validation or errors against truth**. No aggregate score selects a best weight. Aspect ratio is major/minor; orientation is axial (modulo 180°), and is less interpretable for a nearly circular ellipse. Vorticity is in s⁻¹.

In [ ]:
weights = sorted(set(float(w) for w in WEIGHT_SWEEP))
if not weights or not np.isfinite(weights).all() or min(weights)<0:
    raise ValueError('Use finite nonnegative sweep weights.')
inner_sweep = [dw.weighted_fit(observations,reference,f'Inner lambda={w:g}',
    w,SAT_WEIGHT_OUTER,RC_LIMIT_M) for w in weights]
outer_sweep = [dw.weighted_fit(observations,reference,f'Outer lambda={w:g}',
    SAT_WEIGHT_INNER,w,RC_LIMIT_M) for w in weights]
all_fits = comparisons + inner_sweep + outer_sweep
parameters = pd.DataFrame([f['summary'] for f in all_fits])
display(parameters.iloc[len(comparisons):].reindex(columns=
    ['label','satellite_weight_inner','satellite_weight_outer']+columns[1:]))
fig_inner = dw.plot_sweep(inner_sweep,'inner',reference['summary'],evaluation)
plt.show()
fig_outer = dw.plot_sweep(outer_sweep,'outer',reference['summary'],evaluation)
plt.show()

**Outer weight zero is diagnostic:** a finite drifter-only Gaussian radius can be an extrapolation beyond observed radial support. Check `peak_bracketed` and the coverage plot before interpreting it. A numerical success is not evidence that the outer profile is well constrained. Other weights are influence scenarios, not confidence bounds.

Use the results to ask: does the drifter determine a distinct inner solution; does satellite coverage reach the outer decay; and how strongly do radius, rotation and residuals change with the specified weights? One selected day cannot establish robustness over the complete trajectory. Normalisation does not correct ageostrophic motion, drogue effects, depth shear, temporal evolution or correlated samples.

In [ ]:
audit = dw.support(all_fits, observations)
full_residuals = pw.residuals(all_fits,evaluation)
core_residuals = dw.inner_residuals(all_fits,core_evaluation)
with pd.option_context('display.max_rows', len(audit)):
    display(audit)
print('Depths are nominal (15 m drifter, surface satellite); observation uncertainties are not estimated.')
issues = parameters[(~parameters.inner_valid) | ((~parameters.valid)&(parameters.status!='inner_only')) | (parameters.warnings!='')]
if len(issues):
    display(issues.reindex(columns=['label','status','warnings','message']))
else:
    print('All attempted fits passed numerical checks; drifter-only reference is intentionally inner-only.')

In [ ]:
if SAVE_OUTPUTS:
    run_dir = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    run_dir.mkdir(parents=True,exist_ok=False)
    tables = dict(parameters=parameters,full_residuals=full_residuals,core_residuals=core_residuals,
        sample_support=audit,radial_coverage=coverage_table,observations=observations,
        original_combined_history=history,satellite_history=satellite_history)
    for name,table in tables.items():table.to_csv(run_dir/f'{name}.csv',index=False)
    membership = [dict(label=f['summary']['label'],stage=stage,observation_id=i)
        for f in all_fits for stage in ['inner_ids','outer_ids','outer_kept_ids'] for i in f[stage]]
    pd.DataFrame(membership).to_csv(run_dir/'fit_membership.csv',index=False)
    for name,fig in [('coverage',fig_coverage),('inner_weights',fig_inner),('outer_weights',fig_outer)]:
        fig.savefig(run_dir/f'{name}.png',dpi=180,bbox_inches='tight')
    controls = dict(analysis_time=ANALYSIS_TIME,cutoff=DATA_CUTOFF,LON0=LON0,LAT0=LAT0,
        **track_settings,rc_limit=RC_LIMIT_M,inner_weight=SAT_WEIGHT_INNER,
        outer_weight=SAT_WEIGHT_OUTER,weight_sweep=weights)
    files = [Path(vh.__file__),Path(pw.__file__),Path(dw.__file__),Path('04_drifter_satellite_weighting.ipynb')]
    provenance = dict(backend=backend_info,data=data_info,controls=controls,checks=checks,
        core_ids=core_ids,outer_ids=outer_ids,evaluation_ids=evaluation_ids,
        local_sha256={str(p.resolve()):hashlib.sha256(p.read_bytes()).hexdigest() for p in files},
        interpretation='Single-day influence sensitivity and sampling support; not independent validation or uncertainty estimation.')
    (run_dir/'provenance.json').write_text(json.dumps(provenance,indent=2,default=str))
    print('Saved:',run_dir.resolve())
else:
    print('SAVE_OUTPUTS=False: no result files written; tables and figures remain visible.')